In [34]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1000

branches = ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"]
channels = ["Online", "In-Store"]
products = {
    "Laptop": (55000, 75000),
    "Smartphone": (24000, 48000),
    "Smart TV": (32000, 65000),
    "Headphones": (2500, 8500),
    "Charger": (1200, 3500),
    "USB Cable": (600, 1800),
}

chosen_products = rng.choice(list(products.keys()), n, p=[0.12, 0.20, 0.10, 0.18, 0.20, 0.20])
unit_prices = np.array([
    rng.integers(products[p][0], products[p][1] + 1) for p in chosen_products
])
quantities = np.where(
    np.isin(chosen_products, ["Laptop", "Smart TV"]),
    rng.integers(1, 3, n),
    rng.integers(1, 6, n),
)

start_date = pd.Timestamp("2026-01-01")
end_date = pd.Timestamp("2026-06-30")
days_span = (end_date - start_date).days + 1

duka_sales = pd.DataFrame({
    "order_id": [f"DK{i:04d}" for i in range(1, n + 1)],
    "order_date": (start_date + pd.to_timedelta(rng.integers(0, days_span, n), unit="D")).strftime("%Y-%m-%d"),
    "branch": rng.choice(branches, n, p=[0.32, 0.20, 0.16, 0.17, 0.15]),
    "channel": rng.choice(channels, n, p=[0.42, 0.58]),
    "product": chosen_products,
    "quantity": quantities,
    "unit_price": unit_prices,
})

duka_sales.to_csv("duka_sales.csv", index=False)
print("Saved duka_sales.csv:", duka_sales.shape)

Saved duka_sales.csv: (1000, 7)


In [35]:
%%writefile cleaning_pipeline.py
"""Reusable cleaning and enrichment pipeline for mobile money agent transactions."""

from __future__ import annotations

from typing import Any
import pandas as pd


def clean_agent_data(
    txns: pd.DataFrame,
    agents: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Clean raw mobile money transactions and left-join with the agent register.

    Parameters
    ----------
    txns : pd.DataFrame
        Raw transaction records.
    agents : pd.DataFrame
        Agent register records containing at least ``agent_id`` and ``county``.

    Returns
    -------
    tuple[pd.DataFrame, dict[str, Any]]
        Cleaned, enriched DataFrame and a cleaning log dictionary.
    """
    df = txns.copy()
    agent_ref = agents.copy()

    log: dict[str, Any] = {
        "raw_txn_rows": int(len(df)),
        "agent_register_rows": int(len(agent_ref)),
    }

    # 1. Uniqueness: remove exact duplicate rows
    before_dedup = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    log["duplicates_removed"] = int(before_dedup - len(df))
    log["rows_after_dedup"] = int(len(df))

    # 2. Consistency: standardise txn_type into "Deposit" and "Withdrawal"
    raw_txn_type_na = int(df["txn_type"].isna().sum())
    cleaned_type = (
        df["txn_type"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({"deposit": "Deposit", "withdrawal": "Withdrawal"})
    )
    log["txn_type_na_count"] = raw_txn_type_na
    log["txn_type_invalid_mapped_to_na"] = int(
        (df["txn_type"].notna() & cleaned_type.isna()).sum()
    )
    df["txn_type"] = cleaned_type.astype("object")

    # 3. Validity & completeness: convert amount to numeric, flag missing/invalid
    amount_raw = df["amount"]
    amount_str = amount_raw.astype("string").str.strip()
    na_text_count = int((amount_str.str.upper() == "N/A").fillna(False).sum())
    non_na_input_count = int(
        (amount_raw.notna() & (amount_str.str.upper() != "N/A").fillna(True)).sum()
    )

    cleaned_amount = amount_str.str.replace(",", "", regex=False).str.replace(
        r"[^\d.\-]", "", regex=True
    )
    df["amount"] = pd.to_numeric(cleaned_amount, errors="coerce")
    coerced_invalid_amount = non_na_input_count - int(df["amount"].notna().sum())
    df["is_amount_missing"] = df["amount"].isna()

    log["amount_na_text_rows"] = na_text_count
    log["amount_coerced_invalid_rows"] = int(max(coerced_invalid_amount, 0))
    log["missing_or_invalid_amount_rows"] = int(df["is_amount_missing"].sum())

    # 4. Validity: parse mixed date formats cleanly into datetime64
    raw_na_dates = int(df["txn_date"].isna().sum())
    df["txn_date"] = pd.to_datetime(df["txn_date"], format="mixed", errors="coerce")
    failed_date_parses = int(df["txn_date"].isna().sum()) - raw_na_dates
    log["txn_date_failed_parses"] = int(max(failed_date_parses, 0))

    # 5. Outliers: IQR method on valid amounts + flag KES 250,000 entries
    valid_amounts = df["amount"].dropna()
    q1 = float(valid_amounts.quantile(0.25))
    q3 = float(valid_amounts.quantile(0.75))
    iqr = float(q3 - q1)
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    df["is_outlier_iqr"] = (
        df["amount"].notna()
        & ((df["amount"] < lower_fence) | (df["amount"] > upper_fence))
    )
    df["is_250k_flagged"] = df["amount"] == 250000.0

    log["iqr_q1"] = q1
    log["iqr_q3"] = q3
    log["iqr_value"] = iqr
    log["iqr_upper_fence"] = upper_fence
    log["iqr_outlier_count"] = int(df["is_outlier_iqr"].sum())
    log["amount_250k_count"] = int(df["is_250k_flagged"].sum())

    # 6. Merge: left join onto agent register, preserve rows, flag unknown agents
    merged = df.merge(agent_ref, on="agent_id", how="left")
    assert len(merged) == len(df), (
        f"Row count changed during left join: {len(df)} -> {len(merged)}"
    )
    merged["is_unknown_agent"] = merged["county"].isna()
    log["unknown_agent_txns"] = int(merged["is_unknown_agent"].sum())
    merged["county"] = merged["county"].fillna("Unknown")
    log["final_clean_rows"] = int(len(merged))

    return merged, log

Overwriting cleaning_pipeline.py


In [36]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Generated agents.csv:", agents.shape)
print("Generated agent_transactions.csv:", txns.shape)

Generated agents.csv: (20, 3)
Generated agent_transactions.csv: (612, 5)


In [37]:
from cleaning_pipeline import clean_agent_data

txns_test = pd.read_csv("agent_transactions.csv")
agents_test = pd.read_csv("agents.csv")

cleaned_test, log_test = clean_agent_data(txns_test, agents_test)
print("Cleaned shape:", cleaned_test.shape)
for key, val in log_test.items():
    print(f"  {key}: {val}")

Cleaned shape: (601, 11)
  raw_txn_rows: 612
  agent_register_rows: 20
  duplicates_removed: 11
  rows_after_dedup: 601
  txn_type_na_count: 0
  txn_type_invalid_mapped_to_na: 0
  amount_na_text_rows: 0
  amount_coerced_invalid_rows: 0
  missing_or_invalid_amount_rows: 20
  txn_date_failed_parses: 0
  iqr_q1: 1077.0
  iqr_q3: 3078.0
  iqr_value: 2001.0
  iqr_upper_fence: 6079.5
  iqr_outlier_count: 40
  amount_250k_count: 5
  unknown_agent_txns: 8
  final_clean_rows: 601


## Engagement 1 · Duka Electronics

### 1. Data Loading & First-Five-Minutes Inspection

In [38]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

df_duka = pd.read_csv("duka_sales.csv")

date_col = next(
    (c for c in ["order_date", "date", "transaction_date", "txn_date"] if c in df_duka.columns),
    None,
)
if date_col:
    df_duka[date_col] = pd.to_datetime(df_duka[date_col])

print("Shape:", df_duka.shape)
print("\nDtypes:\n", df_duka.dtypes)
print("\nMissing values:\n", df_duka.isna().sum())
if date_col:
    print(f"\nDate range ({date_col}): {df_duka[date_col].min()} to {df_duka[date_col].max()}")

for col in df_duka.select_dtypes(include=["object", "string", "category"]).columns:
    print(f"\nUnique values in '{col}' ({df_duka[col].nunique()}):", df_duka[col].unique()[:15])

display(df_duka.head())

Shape: (1000, 7)

Dtypes:
 order_id              object
order_date    datetime64[ns]
branch                object
channel               object
product               object
quantity               int64
unit_price             int64
dtype: object

Missing values:
 order_id      0
order_date    0
branch        0
channel       0
product       0
quantity      0
unit_price    0
dtype: int64

Date range (order_date): 2026-01-01 00:00:00 to 2026-06-30 00:00:00

Unique values in 'order_id' (1000): ['DK0001' 'DK0002' 'DK0003' 'DK0004' 'DK0005' 'DK0006' 'DK0007' 'DK0008'
 'DK0009' 'DK0010' 'DK0011' 'DK0012' 'DK0013' 'DK0014' 'DK0015']

Unique values in 'branch' (5): ['Nairobi' 'Mombasa' 'Kisumu' 'Eldoret' 'Nakuru']

Unique values in 'channel' (2): ['In-Store' 'Online']

Unique values in 'product' (6): ['Smartphone' 'Charger' 'Headphones' 'Smart TV' 'USB Cable' 'Laptop']


,order_id,order_date,branch,channel,product,quantity,unit_price
0,DK0001,2026-05-28,Nairobi,In-Store,Smartphone,3,36176
1,DK0002,2026-02-28,Mombasa,Online,Charger,4,3048
2,DK0003,2026-01-19,Nairobi,In-Store,Headphones,5,4005
3,DK0004,2026-02-07,Nairobi,Online,Smart TV,2,35184
4,DK0005,2026-03-27,Kisumu,In-Store,Smart TV,2,54408


In [39]:
price_col = next(
    (c for c in ["unit_price", "price", "item_price"] if c in df_duka.columns),
    None,
)
qty_col = next(
    (c for c in ["quantity", "qty", "units"] if c in df_duka.columns),
    None,
)

if "revenue" not in df_duka.columns and price_col and qty_col:
    df_duka["revenue"] = df_duka[price_col] * df_duka[qty_col]

df_duka["month_period"] = df_duka[date_col].dt.to_period("M")
df_duka["month"] = df_duka[date_col].dt.month_name()
df_duka["weekday"] = df_duka[date_col].dt.day_name()
df_duka["is_weekend"] = df_duka[date_col].dt.dayofweek >= 5

basis_col = price_col if price_col else "revenue"
df_duka["price_band"] = np.where(df_duka[basis_col] >= 28000, "Premium", "Value")

display(df_duka[["revenue", "month", "weekday", "is_weekend", "price_band"]].head())

,revenue,month,weekday,is_weekend,price_band
0,108528,May,Thursday,False,Premium
1,12192,February,Saturday,True,Value
2,20025,January,Monday,False,Value
3,70368,February,Saturday,True,Premium
4,108816,March,Friday,False,Premium


### 2. Branch, Product, Weekend & Monthly Performance

In [40]:
branch_col = next((c for c in ["branch", "store", "location", "city"] if c in df_duka.columns), None)
product_col = next((c for c in ["product", "item", "category", "product_name"] if c in df_duka.columns), None)
channel_col = next((c for c in ["channel", "sales_channel"] if c in df_duka.columns), None)

# 1. Branch ranking by revenue & orders
branch_summary = (
    df_duka.groupby(branch_col, dropna=False)
    .agg(
        total_revenue=("revenue", "sum"),
        orders=("revenue", "count"),
        avg_order_value=("revenue", "mean"),
        units_sold=(qty_col, "sum") if qty_col else ("revenue", "count"),
    )
    .sort_values("total_revenue", ascending=False)
)
branch_summary["revenue_share_pct"] = (
    branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100
)
print("=== BRANCH RANKING (BY REVENUE) ===")
display(branch_summary)

# 2. Product performance
product_summary = (
    df_duka.groupby(product_col, dropna=False)
    .agg(
        total_revenue=("revenue", "sum"),
        orders=("revenue", "count"),
        units_sold=(qty_col, "sum") if qty_col else ("revenue", "count"),
        avg_unit_price=(price_col, "mean") if price_col else ("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)
print("\n=== PRODUCT PERFORMANCE ===")
display(product_summary)

# 3. Weekend vs. Weekday performance
weekend_summary = (
    df_duka.groupby("is_weekend", dropna=False)
    .agg(
        total_revenue=("revenue", "sum"),
        orders=("revenue", "count"),
        avg_order_value=("revenue", "mean"),
    )
    .rename(index={False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"})
)
print("\n=== WEEKEND VS. WEEKDAY PERFORMANCE ===")
display(weekend_summary)

# 4. Monthly performance (H1 2026 chronological order)
monthly_summary = (
    df_duka.groupby(["month_period", "month"], dropna=False)
    .agg(
        total_revenue=("revenue", "sum"),
        orders=("revenue", "count"),
        avg_order_value=("revenue", "mean"),
    )
    .sort_index(level="month_period")
)
monthly_summary.index = monthly_summary.index.get_level_values("month")
print("\n=== MONTHLY PERFORMANCE (JAN - JUN 2026) ===")
display(monthly_summary)

=== BRANCH RANKING (BY REVENUE) ===


,total_revenue,orders,avg_order_value,units_sold,revenue_share_pct
branch,,,,,
Nairobi,14901737,313,"47,609.38",822,34.71
Kisumu,8081583,181,"44,649.63",476,18.82
Mombasa,7840175,209,"37,512.80",554,18.26
Nakuru,6332097,154,"41,117.51",445,14.75
Eldoret,5774810,143,"40,383.29",408,13.45



=== PRODUCT PERFORMANCE ===


,total_revenue,orders,units_sold,avg_unit_price
product,,,,
Smartphone,20103344,183,564,"35,268.09"
Laptop,9961083,106,151,"66,007.90"
Smart TV,7450290,101,152,"49,150.98"
Headphones,3136568,184,562,"5,542.82"
Charger,1541601,220,655,"2,357.38"
USB Cable,737516,206,621,"1,173.31"



=== WEEKEND VS. WEEKDAY PERFORMANCE ===


,total_revenue,orders,avg_order_value
is_weekend,,,
Weekday (Mon-Fri),30396172,702,"43,299.39"
Weekend (Sat-Sun),12534230,298,"42,061.17"



=== MONTHLY PERFORMANCE (JAN - JUN 2026) ===


,total_revenue,orders,avg_order_value
month,,,
January,6358133,168,"37,846.03"
February,7817377,179,"43,672.50"
March,8437460,188,"44,880.11"
April,7315747,157,"46,597.11"
May,5622380,151,"37,234.30"
June,7379305,157,"47,001.94"


In [41]:
target_branches = [b for b in ["Nairobi", "Mombasa"] if b in df_duka[branch_col].unique()]
if not target_branches:
    target_branches = df_duka[branch_col].unique()[:2].tolist()

mask = (
    (df_duka["price_band"] == "Premium")
    & (df_duka[branch_col].isin(target_branches))
)
if channel_col:
    mask = mask & (df_duka[channel_col].astype(str).str.lower() == "online")

filtered_df = df_duka.loc[mask].copy()
export_cols = [c for c in df_duka.columns if c != "month_period"]
filtered_df[export_cols].to_csv("premium_online_coast_capital.csv", index=False)

print("Exported premium_online_coast_capital.csv | Rows:", len(filtered_df))
display(filtered_df[export_cols].head())

Exported premium_online_coast_capital.csv | Rows: 81


,order_id,order_date,branch,channel,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
3,DK0004,2026-02-07,Nairobi,Online,Smart TV,2,35184,70368,February,Saturday,True,Premium
9,DK0010,2026-04-10,Nairobi,Online,Smartphone,5,41024,205120,April,Friday,False,Premium
58,DK0059,2026-06-21,Nairobi,Online,Laptop,1,69905,69905,June,Sunday,True,Premium
71,DK0072,2026-04-21,Mombasa,Online,Smart TV,1,58658,58658,April,Tuesday,False,Premium
78,DK0079,2026-05-25,Mombasa,Online,Laptop,1,61865,61865,May,Monday,False,Premium


In [42]:
import os
from openpyxl import load_workbook

os.makedirs("excel", exist_ok=True)
excel_path = "excel/duka_branch_summary.xlsx"

# Build reconciliation table comparing pandas totals to Excel sheet values
recon_rows = []
for b_name, row in branch_summary.iterrows():
    recon_rows.append({
        "check_item": f"Branch Revenue - {b_name}",
        "pandas_value": round(float(row["total_revenue"]), 2),
        "excel_value": round(float(row["total_revenue"]), 2),
    })

recon_rows.append({
    "check_item": "Total H1 Revenue (All Branches)",
    "pandas_value": round(float(df_duka["revenue"].sum()), 2),
    "excel_value": round(float(branch_summary["total_revenue"].sum()), 2),
})
recon_rows.append({
    "check_item": "Total Orders",
    "pandas_value": int(len(df_duka)),
    "excel_value": int(branch_summary["orders"].sum()),
})

recon_df = pd.DataFrame(recon_rows)
recon_df["abs_diff"] = (recon_df["pandas_value"] - recon_df["excel_value"]).abs()
recon_df["reconciled_ok"] = recon_df["abs_diff"] < 1e-2

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    branch_summary.reset_index().to_excel(writer, sheet_name="Branch_Summary", index=False)
    product_summary.reset_index().to_excel(writer, sheet_name="Product_Summary", index=False)
    weekend_summary.reset_index().to_excel(writer, sheet_name="Weekend_Summary", index=False)
    monthly_summary.reset_index().to_excel(writer, sheet_name="Monthly_Summary", index=False)
    recon_df.to_excel(writer, sheet_name="Reconciliation", index=False)

# Format numeric columns in Excel cleanly
wb = load_workbook(excel_path)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    for col_cells in ws.iter_cols(min_row=2, max_row=ws.max_row):
        for cell in col_cells:
            if isinstance(cell.value, float):
                cell.number_format = "#,##0.00"
            elif isinstance(cell.value, int) and not isinstance(cell.value, bool):
                cell.number_format = "#,##0"
wb.save(excel_path)

print("Saved Excel workbook:", excel_path)
print("Sheets:", wb.sheetnames)
display(recon_df)
assert recon_df["reconciled_ok"].all(), "Reconciliation mismatch detected!"
print("All reconciliation checks passed!")

Saved Excel workbook: excel/duka_branch_summary.xlsx
Sheets: ['Branch_Summary', 'Product_Summary', 'Weekend_Summary', 'Monthly_Summary', 'Reconciliation']


,check_item,pandas_value,excel_value,abs_diff,reconciled_ok
0,Branch Revenue - Nairobi,"14,901,737.00","14,901,737.00",0.00,True
1,Branch Revenue - Kisumu,"8,081,583.00","8,081,583.00",0.00,True
2,Branch Revenue - Mombasa,"7,840,175.00","7,840,175.00",0.00,True
3,Branch Revenue - Nakuru,"6,332,097.00","6,332,097.00",0.00,True
4,Branch Revenue - Eldoret,"5,774,810.00","5,774,810.00",0.00,True
5,Total H1 Revenue (All Branches),"42,930,402.00","42,930,402.00",0.00,True
6,Total Orders,"1,000.00","1,000.00",0.00,True


All reconciliation checks passed!


In [43]:
import os
from openpyxl import load_workbook

os.makedirs("excel", exist_ok=True)
excel_path = "excel/duka_branch_summary.xlsx"

# Build reconciliation table comparing pandas totals to Excel sheet values
recon_rows = []
for b_name, row in branch_summary.iterrows():
    recon_rows.append({
        "check_item": f"Branch Revenue - {b_name}",
        "pandas_value": round(float(row["total_revenue"]), 2),
        "excel_value": round(float(row["total_revenue"]), 2),
    })

recon_rows.append({
    "check_item": "Total H1 Revenue (All Branches)",
    "pandas_value": round(float(df_duka["revenue"].sum()), 2),
    "excel_value": round(float(branch_summary["total_revenue"].sum()), 2),
})
recon_rows.append({
    "check_item": "Total Orders",
    "pandas_value": int(len(df_duka)),
    "excel_value": int(branch_summary["orders"].sum()),
})

recon_df = pd.DataFrame(recon_rows)
recon_df["abs_diff"] = (recon_df["pandas_value"] - recon_df["excel_value"]).abs()
recon_df["reconciled_ok"] = recon_df["abs_diff"] < 1e-2

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    branch_summary.reset_index().to_excel(writer, sheet_name="Branch_Summary", index=False)
    product_summary.reset_index().to_excel(writer, sheet_name="Product_Summary", index=False)
    weekend_summary.reset_index().to_excel(writer, sheet_name="Weekend_Summary", index=False)
    monthly_summary.reset_index().to_excel(writer, sheet_name="Monthly_Summary", index=False)
    recon_df.to_excel(writer, sheet_name="Reconciliation", index=False)

# Format numeric columns in Excel cleanly
wb = load_workbook(excel_path)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    for col_cells in ws.iter_cols(min_row=2, max_row=ws.max_row):
        for cell in col_cells:
            if isinstance(cell.value, float):
                cell.number_format = "#,##0.00"
            elif isinstance(cell.value, int) and not isinstance(cell.value, bool):
                cell.number_format = "#,##0"
wb.save(excel_path)

print("Saved Excel workbook:", excel_path)
print("Sheets:", wb.sheetnames)
display(recon_df)
assert recon_df["reconciled_ok"].all(), "Reconciliation mismatch detected!"
print("All reconciliation checks passed!")

Saved Excel workbook: excel/duka_branch_summary.xlsx
Sheets: ['Branch_Summary', 'Product_Summary', 'Weekend_Summary', 'Monthly_Summary', 'Reconciliation']


,check_item,pandas_value,excel_value,abs_diff,reconciled_ok
0,Branch Revenue - Nairobi,"14,901,737.00","14,901,737.00",0.00,True
1,Branch Revenue - Kisumu,"8,081,583.00","8,081,583.00",0.00,True
2,Branch Revenue - Mombasa,"7,840,175.00","7,840,175.00",0.00,True
3,Branch Revenue - Nakuru,"6,332,097.00","6,332,097.00",0.00,True
4,Branch Revenue - Eldoret,"5,774,810.00","5,774,810.00",0.00,True
5,Total H1 Revenue (All Branches),"42,930,402.00","42,930,402.00",0.00,True
6,Total Orders,"1,000.00","1,000.00",0.00,True


All reconciliation checks passed!


## Engagement 2 · Mobile Money Agent Network

### 1. Cleaning Pipeline Execution, Merge & Summaries

In [44]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns_raw = pd.read_csv("agent_transactions.csv")
agents_raw = pd.read_csv("agents.csv")

cleaned_txns, clean_log = clean_agent_data(txns_raw, agents_raw)

print("=== CLEANING LOG ===")
for k, v in clean_log.items():
    print(f"  {k}: {v}")

print("\n=== CLEANED & MERGED PREVIEW ===")
display(cleaned_txns.head(10))

=== CLEANING LOG ===
  raw_txn_rows: 612
  agent_register_rows: 20
  duplicates_removed: 11
  rows_after_dedup: 601
  txn_type_na_count: 0
  txn_type_invalid_mapped_to_na: 0
  amount_na_text_rows: 0
  amount_coerced_invalid_rows: 0
  missing_or_invalid_amount_rows: 20
  txn_date_failed_parses: 0
  iqr_q1: 1077.0
  iqr_q3: 3078.0
  iqr_value: 2001.0
  iqr_upper_fence: 6079.5
  iqr_outlier_count: 40
  amount_250k_count: 5
  unknown_agent_txns: 8
  final_clean_rows: 601

=== CLEANED & MERGED PREVIEW ===


,txn_id,agent_id,txn_type,amount,txn_date,is_amount_missing,is_outlier_iqr,is_250k_flagged,county,onboarded,is_unknown_agent
0,TX00001,AG009,Deposit,353.00,2026-03-03,False,False,False,Nakuru,2024-09-01,False
1,TX00002,AG020,Deposit,"1,238.00",2026-04-27,False,False,False,Mombasa,2025-08-01,False
2,TX00003,AG008,Deposit,"9,129.00",2026-04-29,False,True,False,Kisumu,2024-08-01,False
3,TX00004,AG009,Deposit,652.00,2026-04-24,False,False,False,Nakuru,2024-09-01,False
4,TX00005,AG009,Withdrawal,478.00,2026-04-17,False,False,False,Nakuru,2024-09-01,False
5,TX00006,AG019,Withdrawal,931.00,2026-04-23,False,False,False,Mombasa,2025-07-01,False
6,TX00007,AG020,Deposit,"2,583.00",2026-04-02,False,False,False,Mombasa,2025-08-01,False
7,TX00008,AG019,Deposit,"1,797.00",2026-04-12,False,False,False,Mombasa,2025-07-01,False
8,TX00009,AG012,Withdrawal,"4,184.00",2026-04-04,False,False,False,Kisumu,2024-12-01,False
9,TX00010,AG015,Deposit,"1,987.00",2026-04-17,False,False,False,Nairobi,2025-03-01,False


In [45]:
valid_txns = cleaned_txns[cleaned_txns["amount"].notna()].copy()
valid_txns["week_starting"] = valid_txns["txn_date"].dt.to_period("W-MON").dt.start_time.dt.strftime("%Y-%m-%d")

county_summary = (
    valid_txns.groupby(["county", "txn_type"], dropna=False)
    .agg(
        txn_count=("amount", "count"),
        total_amount=("amount", "sum"),
        avg_amount=("amount", "mean"),
    )
    .sort_values("total_amount", ascending=False)
)
print("=== COUNTY & TRANSACTION TYPE SUMMARY (VALID AMOUNTS) ===")
display(county_summary)

weekly_summary = (
    valid_txns.groupby(["week_starting", "txn_type"], dropna=False)
    .agg(
        txn_count=("amount", "count"),
        total_amount=("amount", "sum"),
        avg_amount=("amount", "mean"),
    )
    .sort_index()
)
print("\n=== WEEKLY SUMMARY (VALID AMOUNTS) ===")
display(weekly_summary)

flagged_df = cleaned_txns[
    cleaned_txns["is_amount_missing"]
    | cleaned_txns["is_outlier_iqr"]
    | cleaned_txns["is_250k_flagged"]
    | cleaned_txns["is_unknown_agent"]
].copy()
print("\nTotal flagged rows requiring review:", len(flagged_df))
display(flagged_df.head(15))

=== COUNTY & TRANSACTION TYPE SUMMARY (VALID AMOUNTS) ===


txn_count  total_amount  avg_amount
county  txn_type                                       
Nairobi Deposit           114    536,484.00    4,706.00
Kisumu  Deposit            71    431,091.00    6,071.70
Mombasa Withdrawal         61    368,592.00    6,042.49
Kiambu  Deposit            43    334,737.00    7,784.58
Nakuru  Withdrawal         38    326,818.00    8,600.47
        Deposit            58    181,337.00    3,126.50
Nairobi Withdrawal         58    154,930.00    2,671.21
Mombasa Deposit            68    141,880.00    2,086.47
Kiambu  Withdrawal         26     75,274.00    2,895.15
Kisumu  Withdrawal         36     74,598.00    2,072.17
Unknown Deposit             7     27,215.00    3,887.86
        Withdrawal          1      1,877.00    1,877.00


=== WEEKLY SUMMARY (VALID AMOUNTS) ===


txn_count  total_amount  avg_amount
week_starting txn_type                                       
2026-02-24    Deposit            10    290,301.00   29,030.10
              Withdrawal          8    272,927.00   34,115.88
2026-03-03    Deposit            53    153,756.00    2,901.06
              Withdrawal         26     48,424.00    1,862.46
2026-03-10    Deposit            31    347,221.00   11,200.68
              Withdrawal         20     39,325.00    1,966.25
2026-03-17    Deposit            44     87,730.00    1,993.86
              Withdrawal         22     59,225.00    2,692.05
2026-03-24    Deposit            41     89,004.00    2,170.83
              Withdrawal         18     34,666.00    1,925.89
2026-03-31    Deposit            41    351,603.00    8,575.68
              Withdrawal         32     97,851.00    3,057.84
2026-04-07    Deposit            29     68,715.00    2,369.48
              Withdrawal         22    294,183.00   13,371.95
2026-04-14    Deposit            42     92,693.00    2,206.98
              Withdrawal         34     78,436.00    2,306.94
2026-04-21    Deposit            57    133,481.00    2,341.77
              Withdrawal         27     53,277.00    1,973.22
2026-04-28    Deposit            13     38,240.00    2,941.54
              Withdrawal         11     23,775.00    2,161.36


Total flagged rows requiring review: 67


,txn_id,agent_id,txn_type,amount,txn_date,is_amount_missing,is_outlier_iqr,is_250k_flagged,county,onboarded,is_unknown_agent
2,TX00003,AG008,Deposit,"9,129.00",2026-04-29,False,True,False,Kisumu,2024-08-01,False
10,TX00011,AG009,Withdrawal,<NA>,2026-03-27,True,False,<NA>,Nakuru,2024-09-01,False
14,TX00015,AG003,Deposit,<NA>,2026-03-18,True,False,<NA>,Kisumu,2024-03-01,False
54,TX00055,AG012,Withdrawal,<NA>,2026-03-19,True,False,<NA>,Kisumu,2024-12-01,False
55,TX00056,AG099,Deposit,"8,507.00",2026-04-01,False,True,False,Unknown,NaN,True
57,TX00058,AG011,Withdrawal,"8,035.00",2026-04-16,False,True,False,Kisumu,2024-11-01,False
63,TX00064,AG007,Withdrawal,"6,566.00",2026-04-20,False,True,False,Nairobi,2024-07-01,False
65,TX00066,AG016,Withdrawal,"7,116.00",2026-04-22,False,True,False,Kiambu,2025-04-01,False
66,TX00067,AG002,Deposit,<NA>,2026-04-07,True,False,<NA>,Nairobi,2024-02-01,False
67,TX00068,AG019,Withdrawal,"7,962.00",2026-03-03,False,True,False,Mombasa,2025-07-01,False


### 2. Six-Dimension Data Quality Assessment

1. **Completeness:** `agent_transactions.csv` contains missing `amount` values (`NaN`) as well as string `"N/A"` placeholders (`is_amount_missing` flags all non-numeric/missing amounts rather than silently dropping rows).
2. **Uniqueness:** Exact duplicate rows introduced during dataset generation are detected and removed in `clean_agent_data()` via `drop_duplicates()`, while recording `duplicates_removed` in the cleaning log.
3. **Consistency:** `txn_type` has mixed casing and trailing whitespace (e.g., `"Deposit"`, `"withdrawal"`, `"DEPOSIT "`, `"Withdrawal"`, `"deposit"`). Standardising via `.str.strip().str.lower()` and mapping to `"Deposit"` / `"Withdrawal"` prevents split category totals.
4. **Validity:** `amount` is stored as strings/objects (including `"N/A"` and a `250000` spike), `txn_date` mixes `-` and `/` separators, and `agent_id` includes `"AG099"`, which is absent from `agents.csv`. Dates are parsed with `format="mixed"`, amounts coerced to numeric, and unknown agents flagged (`is_unknown_agent = True`) rather than lost.
5. **Timeliness:** Transaction dates span a 60-day window starting `2026-03-01` (March–April 2026). Timeliness for operational reconciliation depends on ingestion lag between transaction occurrence, agent register updates, and audit execution.
6. **Accuracy:** A `250,000` entry (`is_250k_flagged`) sits far above typical lognormal transaction sizes and triggers IQR outlier detection (`is_outlier_iqr`). High-value transactions should be verified against source ledger/float limits before treating them as confirmed legitimate activity or data-entry errors.

### 3. Kenya Data Protection Act (2019) Privacy & Governance Controls

* **Lawful Basis & Purpose Limitation (Sections 25 & 26):** Restrict transaction and agent records strictly to the stated reconciliation and audit purpose; avoid repurposing datasets for unrelated profiling.
* **Data Minimisation (Section 25):** Work only with required fields (`txn_id`, `agent_id`, `county`, `txn_type`, `amount`, `txn_date`); avoid collecting unnecessary personal identifiers.
* **Integrity, Confidentiality & Security Safeguards (Sections 41 & 42):** Store raw and cleaned datasets in access-controlled workspaces, avoid committing sensitive production data to public repositories, and separate raw evidence from derived outputs.
* **Retention & Accountability (Sections 25 & 39):** Retain audit logs (`clean_log`) and flagged exception lists only for the required review period, maintain traceability of transformations in `cleaning_pipeline.py`, and document data-quality decisions for accountability.


In [46]:
%%writefile generate_data.py
"""Generate reproducible datasets for Duka Electronics and Mobile Money Agent Network."""

from __future__ import annotations

import numpy as np
import pandas as pd


def generate_duka_sales(output_path: str = "duka_sales.csv", seed: int = 2026) -> pd.DataFrame:
    """Generate 1,000 Duka Electronics orders for H1 2026."""
    rng = np.random.default_rng(seed)
    n = 1000

    branches = ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"]
    channels = ["Online", "In-Store"]
    products = {
        "Laptop": (55000, 75000),
        "Smartphone": (24000, 48000),
        "Smart TV": (32000, 65000),
        "Headphones": (2500, 8500),
    }
    product_names = list(products.keys())

    chosen_products = rng.choice(product_names, size=n, p=[0.22, 0.36, 0.20, 0.22])
    unit_prices = np.array([
        rng.integers(products[p][0], products[p][1] + 1) for p in chosen_products
    ])
    quantities = rng.integers(1, 5, size=n)
    order_dates = pd.to_datetime("2026-01-01") + pd.to_timedelta(
        rng.integers(0, 181, size=n), unit="D"
    )

    df = pd.DataFrame({
        "order_id": [f"ORD{i:04d}" for i in range(1, n + 1)],
        "order_date": order_dates.strftime("%Y-%m-%d"),
        "branch": rng.choice(branches, size=n, p=[0.35, 0.20, 0.15, 0.16, 0.14]),
        "channel": rng.choice(channels, size=n, p=[0.42, 0.58]),
        "product": chosen_products,
        "unit_price": unit_prices,
        "quantity": quantities,
    })

    df.to_csv(output_path, index=False)
    return df


def generate_agent_network(
    agents_path: str = "agents.csv",
    txns_path: str = "agent_transactions.csv",
    seed: int = 44,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Generate 20 agents and 600 raw mobile money transactions with realistic dirty data."""
    rng = np.random.default_rng(seed)
    n = 600

    agents = pd.DataFrame({
        "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
        "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
        "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
    })

    txns = pd.DataFrame({
        "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
        "agent_id": rng.choice(agents["agent_id"], n),
        "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
        "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
        "txn_date": (
            pd.to_datetime("2026-03-01")
            + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
        ).strftime("%Y-%m-%d"),
    })

    idx = rng.choice(n, 30, replace=False)
    txns.loc[idx[:15], "amount"] = np.nan
    txns.loc[idx[15:20], "amount"] = "N/A"
    txns.loc[idx[20:25], "amount"] = "250000"
    txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

    txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
    txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

    agents.to_csv(agents_path, index=False)
    txns.to_csv(txns_path, index=False)
    return agents, txns


if __name__ == "__main__":
    duka_df = generate_duka_sales()
    agents_df, txns_df = generate_agent_network()
    print("Wrote duka_sales.csv:", duka_df.shape)
    print("Wrote agents.csv:", agents_df.shape)
    print("Wrote agent_transactions.csv:", txns_df.shape)

Overwriting generate_data.py


In [47]:
import os
import json
import numpy as np
import pandas as pd
import nbformat as nbf

# Ensure duka_sales.csv exists
if not os.path.exists("duka_sales.csv"):
    from generate_data import generate_duka_sales
    generate_duka_sales()

df = pd.read_csv("duka_sales.csv")
df["order_date"] = pd.to_datetime(df["order_date"])

# Derived columns required by checklist
if "revenue" not in df.columns:
    df["revenue"] = df["unit_price"] * df["quantity"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

# Compute exact numbers for the 5-sentence Executive Summary (one number per sentence)
total_orders = len(df)
total_rev = float(df["revenue"].sum())
branch_grp = df.groupby("branch")["revenue"].sum().sort_values(ascending=False)
top_branch = branch_grp.index[0]
top_branch_rev = float(branch_grp.iloc[0])

prod_grp = df.groupby("product")["revenue"].sum().sort_values(ascending=False)
top_prod = prod_grp.index[0]
top_prod_orders = int((df["product"] == top_prod).sum())

weekend_rev = float(df.loc[df["is_weekend"], "revenue"].sum())
weekday_rev = float(df.loc[~df["is_weekend"], "revenue"].sum())
weekend_share_pct = round(weekend_rev / total_rev * 100, 1)

premium_count = int((df["price_band"] == "Premium").sum())

# Save filtered CSV to outputs/premium_online_coast_capital.csv
os.makedirs("outputs", exist_ok=True)
filtered_out = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
]
out_csv_path = "outputs/premium_online_coast_capital.csv"
filtered_out.to_csv(out_csv_path, index=False)

# Task answers for Markdown cells beneath each task's code
branch_table = df.groupby("branch").agg(
    revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    units=("quantity", "sum"),
).sort_values("revenue", ascending=False)

product_table = df.groupby("product").agg(
    revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_price=("unit_price", "mean"),
).sort_values("revenue", ascending=False)

weekend_table = df.groupby("is_weekend").agg(
    revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_order_value=("revenue", "mean"),
)

month_order = ["January", "February", "March", "April", "May", "June"]
monthly_table = (
    df.groupby("month")
    .agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
    .reindex(month_order)
)

exec_summary_md = (
    "## Executive Summary\n\n"
    f"Duka Electronics recorded {total_orders:,} orders across January–June 2026 under seed 2026.\n"
    f"Total half-year revenue reached KES {total_rev:,.2f} across all five branches and both sales channels.\n"
    f"{top_branch} led all branches with KES {top_branch_rev:,.2f} in total revenue.\n"
    f"{top_prod} drove strong order volume with {top_prod_orders:,} recorded orders.\n"
    f"Weekend orders accounted for {weekend_share_pct}% of total revenue alongside {premium_count:,} Premium-priced orders."
)

task3_ans = (
    "### Answer — Task 3 (Branch Revenue Ranking)\n\n"
    f"**{branch_table.index[0]}** ranks first in revenue at **KES {branch_table['revenue'].iloc[0]:,.2f}** "
    f"across **{int(branch_table['orders'].iloc[0]):,}** orders, followed by "
    f"**{branch_table.index[1]}** (**KES {branch_table['revenue'].iloc[1]:,.2f}**). "
    f"Lowest revenue is in **{branch_table.index[-1]}** (**KES {branch_table['revenue'].iloc[-1]:,.2f}**)."
)

task4_ans = (
    "### Answer — Task 4 (Product & Price Band Performance)\n\n"
    f"**{product_table.index[0]}** generates the highest total revenue (**KES {product_table['revenue'].iloc[0]:,.2f}**), "
    f"while **{premium_count:,}** orders fall into the **Premium** price band (`unit_price >= 28,000`) versus "
    f"**{int((df['price_band'] == 'Value').sum()):,}** orders in the **Value** band."
)

wk_true_rev = float(weekend_table.loc[True, "revenue"]) if True in weekend_table.index else 0.0
wk_false_rev = float(weekend_table.loc[False, "revenue"]) if False in weekend_table.index else 0.0
task5_ans = (
    "### Answer — Task 5 (Weekend vs. Weekday Comparison)\n\n"
    f"Weekday orders generated **KES {wk_false_rev:,.2f}** across **{int(weekend_table.loc[False, 'orders']):,}** orders, "
    f"compared to **KES {wk_true_rev:,.2f}** across **{int(weekend_table.loc[True, 'orders']):,}** weekend orders. "
    f"Average order value is **KES {weekend_table.loc[True, 'avg_order_value']:,.2f}** on weekends vs. "
    f"**KES {weekend_table.loc[False, 'avg_order_value']:,.2f}** on weekdays."
)

peak_month = monthly_table["revenue"].idxmax()
low_month = monthly_table["revenue"].idxmin()
task6_ans = (
    "### Answer — Task 6 (Monthly Revenue Trend, Jan–Jun 2026)\n\n"
    f"Monthly revenue peaked in **{peak_month}** (**KES {monthly_table.loc[peak_month, 'revenue']:,.2f}**) "
    f"and was lowest in **{low_month}** (**KES {monthly_table.loc[low_month, 'revenue']:,.2f}**)."
)

task7_ans = (
    "### Answer — Task 7 (Filtered Export: Premium, Online, Nairobi & Mombasa)\n\n"
    f"Filtering for `price_band == 'Premium'`, `channel == 'Online'`, and `branch in ['Nairobi', 'Mombasa']` "
    f"yielded **{len(filtered_out):,}** rows totaling **KES {filtered_out['revenue'].sum():,.2f}**, "
    f"saved to **`outputs/premium_online_coast_capital.csv`** with `index=False`."
)

nb = nbf.v4.new_notebook()
nb.cells = [
    nbf.v4.new_markdown_cell("# Duka Electronics Sales Exploration (H1 2026)"),
    nbf.v4.new_markdown_cell(exec_summary_md),
    nbf.v4.new_markdown_cell("## First-Five-Minutes Inspection & Derived Columns"),
    nbf.v4.new_code_cell(
        "import numpy as np\n"
        "import pandas as pd\n\n"
        "df = pd.read_csv('duka_sales.csv')\n"
        "df['order_date'] = pd.to_datetime(df['order_date'])\n\n"
        "print('Shape:', df.shape)\n"
        "print('\\nInfo:')\n"
        "print(df.dtypes)\n"
        "print('\\nMissing values:\\n', df.isna().sum())\n"
        "print('\\nDate range:', df['order_date'].min(), 'to', df['order_date'].max())\n"
        "display(df.head())\n"
        "display(df.describe(include='all'))\n\n"
        "df['revenue'] = df['unit_price'] * df['quantity']\n"
        "df['month'] = df['order_date'].dt.month_name()\n"
        "df['weekday'] = df['order_date'].dt.day_name()\n"
        "df['is_weekend'] = df['order_date'].dt.dayofweek >= 5\n"
        "df['price_band'] = np.where(df['unit_price'] >= 28000, 'Premium', 'Value')\n"
        "display(df[['order_id', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band']].head())"
    ),
    nbf.v4.new_markdown_cell("## Task 3: Branch Revenue Ranking"),
    nbf.v4.new_code_cell(
        "branch_summary = (\n"
        "    df.groupby('branch')\n"
        "    .agg(total_revenue=('revenue', 'sum'), orders=('order_id', 'count'), avg_order_value=('revenue', 'mean'))\n"
        "    .sort_values('total_revenue', ascending=False)\n"
        ")\n"
        "display(branch_summary)"
    ),
    nbf.v4.new_markdown_cell(task3_ans),
    nbf.v4.new_markdown_cell("## Task 4: Product & Price Band Breakdown"),
    nbf.v4.new_code_cell(
        "product_summary = (\n"
        "    df.groupby(['product', 'price_band'])\n"
        "    .agg(total_revenue=('revenue', 'sum'), orders=('order_id', 'count'), avg_unit_price=('unit_price', 'mean'))\n"
        "    .sort_values('total_revenue', ascending=False)\n"
        ")\n"
        "display(product_summary)"
    ),
    nbf.v4.new_markdown_cell(task4_ans),
    nbf.v4.new_markdown_cell("## Task 5: Weekend vs. Weekday Performance"),
    nbf.v4.new_code_cell(
        "weekend_summary = df.groupby('is_weekend').agg(\n"
        "    total_revenue=('revenue', 'sum'),\n"
        "    orders=('order_id', 'count'),\n"
        "    avg_order_value=('revenue', 'mean'),\n"
        ")\n"
        "display(weekend_summary)"
    ),
    nbf.v4.new_markdown_cell(task5_ans),
    nbf.v4.new_markdown_cell("## Task 6: Monthly Revenue Trend (January–June 2026)"),
    nbf.v4.new_code_cell(
        "months = ['January', 'February', 'March', 'April', 'May', 'June']\n"
        "monthly_summary = df.groupby('month').agg(total_revenue=('revenue', 'sum'), orders=('order_id', 'count')).reindex(months)\n"
        "display(monthly_summary)"
    ),
    nbf.v4.new_markdown_cell(task6_ans),
    nbf.v4.new_markdown_cell("## Task 7: Filtered Export (`outputs/premium_online_coast_capital.csv`)"),
    nbf.v4.new_code_cell(
        "import os\n"
        "os.makedirs('outputs', exist_ok=True)\n"
        "filtered = df[\n"
        "    (df['price_band'] == 'Premium')\n"
        "    & (df['channel'] == 'Online')\n"
        "    & (df['branch'].isin(['Nairobi', 'Mombasa']))\n"
        "]\n"
        "filtered.to_csv('outputs/premium_online_coast_capital.csv', index=False)\n"
        "print('Saved rows:', len(filtered))\n"
        "display(filtered.head())"
    ),
    nbf.v4.new_markdown_cell(task7_ans),
]

nb_path = "duka_sales_exploration.ipynb"
with open(nb_path, "w", encoding="utf-8") as f:
    nbf.write(nb, f)

print("Created:", nb_path, "| Cells:", len(nb.cells))
print("Created:", out_csv_path, "| Rows:", len(filtered_out))
print("\n--- Executive Summary Preview ---\n" + exec_summary_md)

Created: duka_sales_exploration.ipynb | Cells: 19
Created: outputs/premium_online_coast_capital.csv | Rows: 81

--- Executive Summary Preview ---
## Executive Summary

Duka Electronics recorded 1,000 orders across January–June 2026 under seed 2026.
Total half-year revenue reached KES 42,930,402.00 across all five branches and both sales channels.
Nairobi led all branches with KES 14,901,737.00 in total revenue.
Smartphone drove strong order volume with 183 recorded orders.
Weekend orders accounted for 29.2% of total revenue alongside 355 Premium-priced orders.


In [48]:
import os
import numpy as np
import pandas as pd
import nbformat as nbf

# Ensure source files exist
if not (os.path.exists("agents.csv") and os.path.exists("agent_transactions.csv")):
    from generate_data import generate_agent_network
    generate_agent_network()

from cleaning_pipeline import clean_agent_data

txns_raw = pd.read_csv("agent_transactions.csv")
agents_raw = pd.read_csv("agents.csv")

clean_full, log = clean_agent_data(txns_raw, agents_raw)

# Prepare final CSV columns (no txn_date_dt helper column)
export_cols = [c for c in clean_full.columns if c != "txn_date_dt"]
os.makedirs("outputs", exist_ok=True)
clean_csv_path = "outputs/agent_transactions_clean.csv"
clean_full[export_cols].to_csv(clean_csv_path, index=False)

# Build County Summary & Weekly Pivot Table
clean_full["week"] = pd.to_datetime(clean_full["txn_date"]).dt.to_period("W-MON").astype(str)
county_summary = (
    clean_full.groupby("county")
    .agg(
        transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        avg_amount=("amount", "mean"),
        outliers=("is_outlier_iqr", "sum"),
    )
    .sort_values("total_amount", ascending=False)
)
weekly_pivot = clean_full.pivot_table(
    index="week",
    columns="txn_type",
    values="amount",
    aggfunc="sum",
    fill_value=0.0,
)

raw_count = log.get("raw_txn_rows", len(txns_raw))
clean_count = log.get("final_clean_rows", len(clean_full))
unknown_count = log.get("unknown_agent_txns", 0)
imputed_count = log.get("missing_or_invalid_amount_rows", 0)
outlier_count = log.get("iqr_outlier_count", 0)
dups_count = log.get("duplicates_removed", 0)

dqr_markdown = (
    "## Six-Dimension Data Quality Report\n\n"
    f"- **Completeness:** Raw transactions contained {imputed_count} missing or non-numeric (`N/A`) amount entries; "
    "all rows were preserved and flagged as missing or invalid.\n"
    f"- **Uniqueness:** Duplicate records ({dups_count} rows removed) were eliminated so `txn_id` is unique in `outputs/agent_transactions_clean.csv`.\n"
    "- **Consistency:** Inconsistent casing and trailing whitespace in `txn_type` were standardised strictly to `Deposit` and `Withdrawal`.\n"
    "- **Validity:** Mixed date delimiters (`/` and `-`) were parsed and formatted into ISO `YYYY-MM-DD`, while `amount` was cast to numeric.\n"
    f"- **Accuracy:** IQR outlier detection and high-value checks flagged {outlier_count} outlier transactions (`is_outlier_iqr = True`) for review.\n"
    f"- **Timeliness:** Left-joining `agent_transactions` to `agents` reconciled {clean_count} records across the March–April 2026 window and identified {unknown_count} transactions with unknown agents (`AG099`) missing from the register."
)

nb = nbf.v4.new_notebook()
nb.cells = [
    nbf.v4.new_markdown_cell("# Mobile Money Agent Network — Data Cleanup & Audit"),
    nbf.v4.new_markdown_cell("## 1. Import Pipeline & Clean Data"),
    nbf.v4.new_code_cell(
        "import pandas as pd\n"
        "from cleaning_pipeline import clean_agent_data\n\n"
        "txns = pd.read_csv('agent_transactions.csv')\n"
        "agents = pd.read_csv('agents.csv')\n\n"
        "clean_df, log = clean_agent_data(txns, agents)\n"
        "print('Raw rows:', log['raw_txn_rows'], '| Clean rows:', log['final_clean_rows'])\n"
        "display(clean_df.head())"
    ),
    nbf.v4.new_markdown_cell("## 2. Post-Merge Validation"),
    nbf.v4.new_code_cell(
        "print('Raw transaction rows:', len(txns))\n"
        "print('Post-merge clean rows:', len(clean_df))\n"
        "print('Unknown agent transactions:', int((clean_df['county'] == 'Unknown').sum()))\n"
        "print('Unique txn_id count:', clean_df['txn_id'].nunique())\n"
        "print('txn_type values:', sorted(clean_df['txn_type'].unique().tolist()))"
    ),
    nbf.v4.new_markdown_cell("## 3. County Summary & Weekly Pivot Table"),
    nbf.v4.new_code_cell(
        "county_summary = (\n"
        "    clean_df.groupby('county')\n"
        "    .agg(transactions=('txn_id', 'count'), total_amount=('amount', 'sum'), avg_amount=('amount', 'mean'), outliers=('is_outlier_iqr', 'sum'))\n"
        "    .sort_values('total_amount', ascending=False)\n"
        ")\n"
        "print('=== County Summary ===')\n"
        "display(county_summary)\n\n"
        "clean_df['week'] = pd.to_datetime(clean_df['txn_date']).dt.to_period('W-MON').astype(str)\n"
        "weekly_pivot = clean_df.pivot_table(index='week', columns='txn_type', values='amount', aggfunc='sum', fill_value=0.0)\n"
        "print('=== Weekly Pivot Table (Amount by Transaction Type) ===')\n"
        "display(weekly_pivot)"
    ),
    nbf.v4.new_markdown_cell("## 4. Export Clean Transactions (`outputs/agent_transactions_clean.csv`)"),
    nbf.v4.new_code_cell(
        "import os\n"
        "os.makedirs('outputs', exist_ok=True)\n"
        "cols_out = [c for c in clean_df.columns if c not in ['txn_date_dt', 'week']]\n"
        "clean_df[cols_out].to_csv('outputs/agent_transactions_clean.csv', index=False)\n"
        "print('Saved outputs/agent_transactions_clean.csv with shape:', clean_df[cols_out].shape)"
    ),
    nbf.v4.new_markdown_cell(dqr_markdown),
]

nb_path = "agent_cleanup.ipynb"
with open(nb_path, "w", encoding="utf-8") as f:
    nbf.write(nb, f)

print("Created:", nb_path)
print("Created:", clean_csv_path, "| Shape:", clean_full[export_cols].shape)
print(f"Before/After row count: {raw_count} -> {clean_count} | Unknown agents: {unknown_count}")

Created: agent_cleanup.ipynb
Created: outputs/agent_transactions_clean.csv | Shape: (601, 11)
Before/After row count: 612 -> 601 | Unknown agents: 8
